# Robustness check — rolling temporal splits, neural network (SPX)

Same design and same three splits as `robustness_spx.ipynb` (XGBoost), applied to the
neural network. Reuses the fixed architecture already selected for the main analysis
(`hidden_layer_sizes=(10,)`, logistic activation, Adam) rather than re-running the
hidden-unit search for each split — consistent with the main text's disclosure that the
network's units were selected once, not re-tuned, for computational reasons.

- Same five inputs, same v4 target (`log(C/K)`) as the main model.
- Inputs are standardized on each split's own train+val set (never on data the model
  will be tested on).
- **Before running**: confirm `SPX_cleaned.csv` covers the full sample (7,688,150 rows,
  31 Aug 2020 to 29 Aug 2025) — the assert below stops execution otherwise.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import joblib


In [ ]:
spx = pd.read_csv("SPX_cleaned.csv")
spx["date"] = pd.to_datetime(spx["date"])
spx["exdate"] = pd.to_datetime(spx["exdate"])

print(spx.shape)
print(spx["date"].min(), spx["date"].max())
spx.head()


(7688150, 17)
2020-08-31 00:00:00 2025-08-29 00:00:00


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate,days_to_maturity,T,q
0,108105,2020-08-31,2020-09-18,100.0,3402.5,3408.2,0,611,133485146,3500.31,35.003100,3405.35,0.111899,0.002148,18,0.049315,0.01589
1,108105,2020-08-31,2020-09-18,1000.0,2503.2,2508.0,0,37663,130915017,3500.31,3.500310,2505.60,0.111899,0.002148,18,0.049315,0.01589
2,108105,2020-08-31,2020-09-18,1100.0,2403.4,2407.9,0,96,130915018,3500.31,3.182100,2405.65,0.111899,0.002148,18,0.049315,0.01589
3,108105,2020-08-31,2020-09-18,1200.0,2303.5,2307.9,0,19,129372797,3500.31,2.916925,2305.70,0.111899,0.002148,18,0.049315,0.01589
4,108105,2020-08-31,2020-09-18,1250.0,2253.7,2258.0,0,20,129372798,3500.31,2.800248,2255.85,0.111899,0.002148,18,0.049315,0.01589


In [ ]:
assert spx.shape[0] > 7_000_000, f"Unexpected row count: {spx.shape[0]:,} — wrong file?"
assert spx["date"].max().year == 2025, f"Unexpected max date: {spx['date'].max()} — wrong file?"
print("OK — full SPX sample loaded.")


OK — full SPX sample loaded.


## Shared settings (identical to the main NN notebook)

In [ ]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for BS and each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS),
                      "BS": np.abs(df["price"].values - df["bs_price"].values)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

feature_cols = ["moneyness (S/K)", "T", "rate", "q", "volatility"]
hidden_units = 10  # selected for the main SPX model; reused as-is, not re-tuned per split


## 1. Black-Scholes benchmark

Identical formula and inputs as the main notebook. BS price does not depend on the
split, so it is computed once on the full dataset.


In [ ]:
d1 = (
    np.log(spx["close"] / spx["strike_price"])
    + (spx["rate"] - spx["q"] + 0.5 * spx["volatility"] ** 2) * spx["T"]
) / (spx["volatility"] * np.sqrt(spx["T"]))

d2 = d1 - spx["volatility"] * np.sqrt(spx["T"])

spx["bs_price"] = (
    spx["close"] * np.exp(-spx["q"] * spx["T"]) * norm.cdf(d1)
    - spx["strike_price"] * np.exp(-spx["rate"] * spx["T"]) * norm.cdf(d2)
)

spx["bs_price"].describe()


,bs_price
count,7.688150e+06
mean,4.168183e+02
std,6.769914e+02
min,0.000000e+00
25%,3.834214e+01
50%,1.692638e+02
75%,4.849242e+02
max,6.298850e+03


## 2. Three rolling, non-overlapping splits

Same date boundaries used for the XGBoost robustness check on SPX and for AAPL, for
direct comparability.


In [ ]:
splits = {
    "C": {
        "train": ("2020-08-31", "2021-08-31"),
        "val":   ("2021-09-01", "2022-01-01"),
        "test":  ("2022-01-02", "2022-04-30"),
    },
    "B": {
        "train": ("2022-05-01", "2023-05-01"),
        "val":   ("2023-05-02", "2023-09-01"),
        "test":  ("2023-09-02", "2023-12-31"),
    },
    "A": {
        "train": ("2024-01-01", "2025-01-01"),
        "val":   ("2025-01-02", "2025-05-01"),
        "test":  ("2025-05-02", "2025-08-29"),
    },
}

for name, s in splits.items():
    print(name, s)


C {'train': ('2020-08-31', '2021-08-31'), 'val': ('2021-09-01', '2022-01-01'), 'test': ('2022-01-02', '2022-04-30')}
B {'train': ('2022-05-01', '2023-05-01'), 'val': ('2023-05-02', '2023-09-01'), 'test': ('2023-09-02', '2023-12-31')}
A {'train': ('2024-01-01', '2025-01-01'), 'val': ('2025-01-02', '2025-05-01'), 'test': ('2025-05-02', '2025-08-29')}


## 3. Fit and evaluate each split

Same recipe as the main notebook's final NN: standardize inputs on train+val, fit on
`log(C/K)`, evaluate on test, convert back with `exp(pred) * K`. The scaler and the
network are both refit from scratch on each split's own train+val set.


In [ ]:
def fit_eval_split_nn(df, split_dates, label):
    tr = df[(df["date"] >= split_dates["train"][0]) & (df["date"] <= split_dates["train"][1])]
    va = df[(df["date"] >= split_dates["val"][0])   & (df["date"] <= split_dates["val"][1])]
    te = df[(df["date"] >= split_dates["test"][0])  & (df["date"] <= split_dates["test"][1])]
    tv = pd.concat([tr, va]).sort_values("date").reset_index(drop=True)
    te = te.sort_values("date").reset_index(drop=True)

    print(f"[{label}] train {len(tr):,} | val {len(va):,} | train+val {len(tv):,} | test {len(te):,}")
    print(f"[{label}] test window: {te['date'].min()} to {te['date'].max()}")

    X_tv = tv[feature_cols].values
    K_tv = tv["strike_price"].values
    y_tv_log = np.log(tv["price"].values / K_tv)

    scaler = StandardScaler().fit(X_tv)
    X_tv_scaled = scaler.transform(X_tv)

    model = MLPRegressor(
        hidden_layer_sizes=(hidden_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42,
    )
    model.fit(X_tv_scaled, y_tv_log)

    X_te = te[feature_cols].values
    K_te = te["strike_price"].values
    X_te_scaled = scaler.transform(X_te)
    test_pred = np.exp(model.predict(X_te_scaled)) * K_te

    return te, test_pred, model, scaler

results = {}
for name, dates in splits.items():
    te, pred, model, scaler = fit_eval_split_nn(spx, dates, f"SPX-NN-{name}")
    results[name] = {"test": te, "pred": pred, "model": model, "scaler": scaler}


[SPX-NN-C] train 1,229,480 | val 476,140 | train+val 1,705,620 | test 481,213
[SPX-NN-C] test window: 2022-01-03 00:00:00 to 2022-04-29 00:00:00
[SPX-NN-B] train 1,455,378 | val 519,178 | train+val 1,974,556 | test 507,984
[SPX-NN-B] test window: 2023-09-05 00:00:00 to 2023-12-29 00:00:00
[SPX-NN-A] train 1,762,789 | val 625,038 | train+val 2,387,827 | test 630,950
[SPX-NN-A] test window: 2025-05-02 00:00:00 to 2025-08-29 00:00:00


## 4. Per-bucket MAE, each split

In [ ]:
bucket_tables = {}
for name, r in results.items():
    bucket_tables[name] = bucket_mae(r["test"], {"NN": r["pred"]})
    print(f"--- Split {name} ---")
    print(bucket_tables[name])
    print()


--- Split C ---
                    n         BS           NN
bucket                                       
Deep OTM        15912   7.109710     1.558971
OTM            132956  15.045603    17.717105
ATM            177751  19.556221    91.668022
ITM            100939  29.135890   270.246952
Deep ITM        40252  28.615935   278.521020
Very Deep ITM   10072   7.474022   176.634237
Extreme ITM      3331   5.795984  1066.215624
ALL            481213  20.317513   129.868881

--- Split B ---
                    n         BS           NN
bucket                                       
Deep OTM         8527   0.776898     0.627801
OTM             89300   5.968536     6.402730
ATM            224678  14.321074    31.692418
ITM            117554  18.658930    62.031399
Deep ITM        50532  12.473524   152.356240
Very Deep ITM   12013   2.799430   623.968967
Extreme ITM      5380   2.392322  1265.740632
ALL            507984  13.046650    72.825196

--- Split A ---
                    n         

## 5. Side-by-side comparison

In [ ]:
summary = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    summary[f"BS_{name}"] = t["BS"]
    summary[f"NN_{name}"] = t["NN"]

summary.round(3)


,BS_C,NN_C,BS_B,NN_B,BS_A,NN_A
Deep OTM,7.110,1.559,0.777,0.628,45.055,1.386
OTM,15.046,17.717,5.969,6.403,76.943,15.188
ATM,19.556,91.668,14.321,31.692,57.170,50.358
ITM,29.136,270.247,18.659,62.031,46.292,125.864
Deep ITM,28.616,278.521,12.474,152.356,17.195,440.814
Very Deep ITM,7.474,176.634,2.799,623.969,4.167,2492.156
Extreme ITM,5.796,1066.216,2.392,1265.741,3.554,2486.688
ALL,20.318,129.869,13.047,72.825,50.000,176.102


In [ ]:
# Same comparison expressed as a ratio (NN error / BS error): <1 means NN wins
ratio = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    ratio[name] = t["NN"] / t["BS"]

ratio.round(3)


,C,B,A
Deep OTM,0.219,0.808,0.031
OTM,1.178,1.073,0.197
ATM,4.687,2.213,0.881
ITM,9.275,3.324,2.719
Deep ITM,9.733,12.214,25.635
Very Deep ITM,23.633,222.891,598.114
Extreme ITM,183.958,529.084,699.697
ALL,6.392,5.582,3.522


## 6. Convergence check

Confirms whether the network actually converged on each split's (much smaller) training
set, rather than stopping early for lack of data — relevant given the shorter training
window here (~12 months) than in the main analysis (~3.5 years).


In [ ]:
for name, r in results.items():
    m = r["model"]
    print(f"Split {name}: n_iter_={m.n_iter_} (max_iter={m.max_iter}), "
          f"final loss={m.loss_:.6f}, converged={m.n_iter_ < m.max_iter}")


Split C: n_iter_=147 (max_iter=2000), final loss=0.071117, converged=True
Split B: n_iter_=120 (max_iter=2000), final loss=0.075629, converged=True
Split A: n_iter_=99 (max_iter=2000), final loss=0.093235, converged=True


## 7. Save results

In [ ]:
joblib.dump(
    {"splits": splits, "hidden_units": hidden_units, "summary": summary, "ratio": ratio,
     "bucket_tables": bucket_tables},
    "robustness_nn_spx_results.pkl"
)


['robustness_nn_spx_results.pkl']